# Легкое дообучение LLM: fine-tuning, LoRA, QLoRA, квантизация и Unsloth

Главная идея: не обучать всю большую модель, а обучать небольшой набор адаптеров поверх квантизированной предобученной модели.

Используем:

- `Hugging Face Transformers`;
- `datasets`;
- `PEFT`;
- `bitsandbytes`;
- `Unsloth` как ускоренную альтернативу;
- маленькую instruct-модель `Qwen/Qwen2.5-0.5B-Instruct`;
- русскоязычный instruction dataset `IlyaGusev/ru_turbo_alpaca`.

Блокнот рассчитан на простой GPU. Для CPU он полезен как конспект, но обучение будет медленным.

## План занятия

| Этап | Тема |
|---|---|
| 1 | Что такое fine-tuning |
| 2 | Полное дообучение vs PEFT |
| 3 | LoRA и QLoRA |
| 4 | 4-bit квантизация через bitsandbytes |
| 5 | Подготовка instruction dataset |
| 6 | Дообучение через Hugging Face + PEFT |
| 7 | Инференс с LoRA-адаптером |
| 8 | Ускоренная альтернатива через Unsloth |
| 9 | Сохранение, слияние и практические советы |

Задача занятия:

$$
\text{base LLM} + \text{instruction dataset} \rightarrow \text{модель, лучше следующая инструкциям}.
$$

## 1. Теория: что такое fine-tuning

Предобученная языковая модель уже знает общие закономерности языка:

$$
P_\theta(x_t \mid x_1, \ldots, x_{t-1}).
$$

Fine-tuning адаптирует модель к конкретному формату данных или задаче.

Примеры:

- отвечать в нужном стиле;
- следовать инструкциям;
- писать код в нужном формате;
- работать с предметной областью;
- извлекать данные по шаблону.

Полное дообучение обновляет все параметры модели. Для LLM это дорого:

$$
\theta \leftarrow \theta + \Delta \theta.
$$

PEFT обновляет только небольшую добавку:

$$
\theta \text{ frozen}, \quad \phi \text{ trainable}.
$$

## 2. LoRA

LoRA означает **Low-Rank Adaptation**.

Вместо обучения всей матрицы весов $W$ мы обучаем низкоранговую добавку:

$$
W' = W + \Delta W,
$$

$$
\Delta W = BA,
$$

где:

- $W$ — замороженные веса предобученной модели;
- $A$ и $B$ — маленькие обучаемые матрицы;
- $r$ — ранг LoRA, обычно 4, 8, 16, 32.

Если $W \in \mathbb{R}^{d \times k}$, то полное дообучение обучает $d \cdot k$ параметров.
LoRA обучает:

$$
r(d + k).
$$

Это радикально снижает потребление памяти.

## 3. Квантизация и QLoRA

Квантизация уменьшает точность хранения весов.

Например:

- `float16`: 16 бит на параметр;
- `int8`: 8 бит;
- `4-bit`: 4 бита.

QLoRA:

1. загружает базовую модель в 4-bit;
2. замораживает квантизированные веса;
3. обучает LoRA-адаптеры в более высокой точности.

Практический результат: можно дообучать модели, которые не поместились бы в память GPU при полном fine-tuning.

## 4. Установка библиотек

Для обычного пути нужны:

- `transformers`;
- `datasets`;
- `peft`;
- `accelerate`;
- `bitsandbytes`;
- `trl`.

Для Unsloth — отдельная установка `unsloth`.

Важно: `bitsandbytes` и Unsloth лучше всего раскрываются на NVIDIA GPU. На Windows иногда удобнее запускать такие ноутбуки в WSL, Linux или Colab.

In [ ]:
%pip install -q -U transformers datasets peft accelerate bitsandbytes trl sentencepiece

### Установка Unsloth

Эта ячейка необязательная. Выполняйте ее, если хотите пройти часть с Unsloth.

В официальной документации для базовой установки используется:

```bash
pip install unsloth
```

Если окружение нестандартное, лучше свериться с документацией Unsloth под вашу версию CUDA/PyTorch.

In [1]:
import os
import gc
import inspect
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)


SEED = 42


def set_global_seed(seed):
    """
    Фиксирует генераторы случайных чисел.

    Аргументы:
        seed: целое число для инициализации генераторов.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 5070 Ti
bf16 supported: True


## 5. Конфигурация эксперимента

Выбранная модель маленькая: `Qwen/Qwen2.5-0.5B-Instruct`.

Почему она удобна для занятия:

- меньше 1B параметров;
- умеет instruction/chat формат;
- поддерживает русский язык;
- подходит для демонстрации LoRA/QLoRA на ограниченной GPU.

Датасет: `IlyaGusev/ru_turbo_alpaca`, русскоязычные instruction-output пары.

Для быстрой демонстрации берем маленькую подвыборку. Для настоящего эксперимента увеличьте `TRAIN_EXAMPLES`.

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME = "d0rj/alpaca-cleaned-ru"

MAX_LENGTH = 512
TRAIN_EXAMPLES = 1200
EVAL_EXAMPLES = 200

OUTPUT_DIR = "qwen_05b_ru_alpaca_lora"
ADAPTER_DIR = "qwen_05b_ru_alpaca_lora_adapter"

USE_4BIT = True

## 6. Загрузка и просмотр датасета

Instruction dataset обычно содержит:

- `instruction` — задание;
- `input` — дополнительный контекст;
- `output` — правильный ответ.

Мы превратим эти поля в chat-формат.

In [3]:
raw_dataset = load_dataset(DATASET_NAME)
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 51760
    })
})

In [4]:
train_raw = raw_dataset["train"]

print(train_raw.column_names)
pd.DataFrame(train_raw.select(range(3))).head()

['input', 'instruction', 'output']


,input,instruction,output
0,,"Дайте три совета, как оставаться здоровым.",1. Соблюдайте сбалансированную и питательную д...
1,,Какие три основных цвета?,"Три основных цвета — красный, синий и желтый. ..."
2,,Опишите строение атома.,Атом является основным строительным блоком все...


## 7. Форматирование instruction данных

Для instruct/chat моделей важно подавать данные в формате диалога.

Будем использовать `tokenizer.apply_chat_template`, если у токенизатора есть chat template.

Каждый пример превратим в:

```text
system: Ты полезный ассистент...
user: инструкция + вход
assistant: ответ
```

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_user_message(example):
    """
    Собирает пользовательское сообщение из instruction и input.

    Аргументы:
        example: один пример датасета.

    Возвращает:
        Строку с заданием пользователя.
    """
    instruction = str(example.get("instruction", "")).strip()
    input_text = str(example.get("input", "")).strip()

    if input_text and input_text.lower() != "nan":
        return f"{instruction}\n\nДополнительный контекст:\n{input_text}"

    return instruction


def format_chat_example(example):
    """
    Преобразует пример instruction dataset в единую строку для SFT.

    Аргументы:
        example: один пример с полями instruction, input, output.

    Возвращает:
        Словарь с полем text.
    """
    messages = [
        {
            "role": "system",
            "content": "Ты полезный русскоязычный ассистент. Отвечай ясно, кратко и по делу.",
        },
        {
            "role": "user",
            "content": build_user_message(example),
        },
        {
            "role": "assistant",
            "content": str(example.get("output", "")).strip(),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


formatted_dataset = train_raw.map(format_chat_example, remove_columns=train_raw.column_names)

formatted_dataset[0]["text"][:1000]

'<|im_start|>system\nТы полезный русскоязычный ассистент. Отвечай ясно, кратко и по делу.<|im_end|>\n<|im_start|>user\nДайте три совета, как оставаться здоровым.<|im_end|>\n<|im_start|>assistant\n1. Соблюдайте сбалансированную и питательную диету. Убедитесь, что в ваш рацион входят разнообразные фрукты и овощи, нежирный белок, цельнозерновые продукты и полезные жиры. Это помогает обеспечить ваш организм необходимыми питательными веществами для оптимального функционирования и может помочь предотвратить хронические заболевания.\n\n2. Занимайтесь регулярной физической активностью. Упражнения имеют решающее значение для поддержания крепких костей, мышц и здоровья сердечно-сосудистой системы. Старайтесь уделять не менее 150 минут умеренным аэробным упражнениям или 75 минут интенсивным упражнениям каждую неделю.\n\n3. Высыпайтесь. Достаточное количество качественного сна имеет решающее значение для физического и психического благополучия. Он помогает регулировать настроение, улучшать когнити

In [6]:
small_dataset = formatted_dataset.shuffle(seed=SEED).select(range(TRAIN_EXAMPLES + EVAL_EXAMPLES))
split_dataset = small_dataset.train_test_split(test_size=EVAL_EXAMPLES, seed=SEED)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['text'],
    num_rows: 1200
})
Dataset({
    features: ['text'],
    num_rows: 200
})


## 8. Токенизация

Для causal language modeling модель учится предсказывать следующий токен.

В простом варианте:

$$
\text{labels} = \text{input\_ids}.
$$

То есть модель обучается воспроизводить всю строку: system, user и assistant.

В production часто маскируют loss на prompt-части и считают loss только по ответу assistant. Для учебного блокнота оставим простой и прозрачный вариант.

In [7]:
def tokenize_examples(batch):
    """
    Токенизирует batch текстов.

    Аргументы:
        batch: batch из Hugging Face Dataset с полем text.

    Возвращает:
        input_ids и attention_mask.
    """
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized_train = train_dataset.map(tokenize_examples, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_examples, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

tokenized_train[0].keys()

dict_keys(['input_ids', 'attention_mask'])

## 9. Загрузка модели с 4-bit квантизацией

`BitsAndBytesConfig` управляет квантизацией.

Для QLoRA часто используют:

- `load_in_4bit=True`;
- `bnb_4bit_quant_type="nf4"`;
- `bnb_4bit_use_double_quant=True`;
- `bnb_4bit_compute_dtype=torch.bfloat16` или `torch.float16`.

Если 4-bit не поддерживается в вашем окружении, можно поставить `USE_4BIT = False`.

In [8]:
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
else:
    quantization_config = None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model) if USE_4BIT else model

print("Модель загружена.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Модель загружена.


## 10. Добавление LoRA-адаптеров через PEFT

Для Qwen-подобных transformer-моделей часто адаптируют проекции attention и MLP:

- `q_proj`;
- `k_proj`;
- `v_proj`;
- `o_proj`;
- `gate_proj`;
- `up_proj`;
- `down_proj`.

Гиперпараметры:

- `r` — ранг LoRA;
- `lora_alpha` — масштаб LoRA;
- `lora_dropout` — регуляризация.

In [9]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


## 11. Дообучение

Сделаем короткое обучение:

- 1 эпоха;
- маленький batch size;
- gradient accumulation;
- небольшой learning rate.

Для полноценного результата увеличьте:

- `TRAIN_EXAMPLES`;
- `num_train_epochs`;
- `max_steps`;
- качество и чистоту данных.

In [10]:
def make_training_arguments():
    """
    Создает TrainingArguments с учетом разных версий transformers.

    Возвращает:
        Объект TrainingArguments.
    """
    signature = inspect.signature(TrainingArguments.__init__).parameters
    eval_argument = "eval_strategy" if "eval_strategy" in signature else "evaluation_strategy"

    return TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        max_steps=-1,
        warmup_ratio=0.03,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        gradient_checkpointing=True,
        fp16=(compute_dtype == torch.float16),
        bf16=(compute_dtype == torch.bfloat16),
        report_to="none",
        optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
        **{eval_argument: "steps"},
        eval_steps=50,
    )


training_args = make_training_arguments()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
50,1.365880,1.333645
75,1.357823,1.325958


TrainOutput(global_step=75, training_loss=1.4757383664449055, metrics={'train_runtime': 160.7527, 'train_samples_per_second': 7.465, 'train_steps_per_second': 0.467, 'total_flos': 952877907801600.0, 'train_loss': 1.4757383664449055, 'epoch': 1.0})

## 12. Сохранение LoRA-адаптера

После PEFT/LoRA обычно сохраняют не всю базовую модель, а только адаптер.

Это удобно:

- меньше размер файлов;
- можно подключать адаптер к той же базовой модели;
- можно хранить несколько адаптеров под разные задачи.

In [11]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Адаптер сохранен в:", ADAPTER_DIR)

Адаптер сохранен в: qwen_05b_ru_alpaca_lora_adapter


## 13. Инференс после дообучения

Сгенерируем ответ на русскую инструкцию.

Если вы не запускали обучение, эта ячейка будет использовать текущую модель с LoRA-слоями в их текущем состоянии.

In [12]:
def generate_answer(model, tokenizer, instruction, max_new_tokens=160):
    """
    Генерирует ответ модели в chat-формате.

    Аргументы:
        model: causal language model.
        tokenizer: токенизатор модели.
        instruction: пользовательская инструкция.
        max_new_tokens: максимальное число новых токенов.

    Возвращает:
        Строку ответа.
    """
    messages = [
        {
            "role": "system",
            "content": "Ты полезный русскоязычный ассистент. Отвечай ясно, кратко и по делу.",
        },
        {
            "role": "user",
            "content": instruction,
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [13]:
print(generate_answer(
    model,
    tokenizer,
    "Объясни простыми словами, что такое LoRA в дообучении языковых моделей.",
))

LoRA (Long Range Association) в дообучении языковых моделей — это технология, используемая для улучшения и улучшенения точности и производительности модели, которая уже обучена на некотором наборе данных. В LoRA, модель может быть обучена на наборе данных, который она уже обработала и распознала. Затем модель может использовать этот набор данных, чтобы улучшить ее предсказание и предсказания. Таким образом, LoRA может увеличить точность и производительность модели, что позволяет естественнее обучаться и лучше понимать язык.


## 14. Загрузка сохраненного адаптера

Чтобы использовать адаптер позже:

1. загрузите базовую модель;
2. подключите LoRA через `PeftModel.from_pretrained`.

Ниже ячейка-шаблон. Ее удобно использовать после перезапуска ноутбука.

In [14]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
tuned_model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

# Часть B. Unsloth

Unsloth — библиотека для ускоренного и экономного fine-tuning LLM.

В этом разделе показан аналогичный workflow:

1. загрузить 4-bit модель через `FastLanguageModel`;
2. добавить LoRA;
3. обучить через `SFTTrainer`;
4. включить быстрый inference.

Эта часть необязательная, но полезна для сравнения с обычным Hugging Face + PEFT.

## 15. Unsloth: загрузка модели

Для Unsloth используем готовую 4-bit модель:

`unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit`.

In [15]:
from unsloth import FastLanguageModel

UNSLOTH_MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

unsloth_model, unsloth_tokenizer = FastLanguageModel.from_pretrained(
    model_name=UNSLOTH_MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if unsloth_tokenizer.pad_token is None:
    unsloth_tokenizer.pad_token = unsloth_tokenizer.eos_token

/home/dmitry/.local/lib/python3.12/site-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[unsloth.import_fixes|WARNING]Unsloth: Detected broken vLLM binary extension; disabling vLLM imports and continuing import.
Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.20.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 16. Unsloth: добавление LoRA

`FastLanguageModel.get_peft_model` добавляет LoRA-адаптеры и включает оптимизации Unsloth.

In [16]:
unsloth_model = FastLanguageModel.get_peft_model(
    unsloth_model,
    r=8,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.10 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 17. Unsloth: SFTTrainer

Для supervised fine-tuning удобно использовать `trl.SFTTrainer`.

Обратите внимание: API `trl` может меняться между версиями. Если ячейка ругается на аргументы, проверьте версию `trl` и документацию.

In [17]:
from trl import SFTConfig, SFTTrainer

unsloth_training_args = SFTConfig(
    output_dir="unsloth_qwen_05b_ru_alpaca_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_steps=100,
    logging_steps=10,
    save_strategy="no",
    dataset_text_field="text",
    max_seq_length=MAX_LENGTH,
    packing=False,
    report_to="none",
)

unsloth_trainer = SFTTrainer(
    model=unsloth_model,
    tokenizer=unsloth_tokenizer,
    args=unsloth_training_args,
    train_dataset=train_dataset,
)

unsloth_trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/1200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,200 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 4,399,104 of 498,431,872 (0.88% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.996317
20,1.599542
30,1.460997
40,1.408967
50,1.341006
60,1.389877
70,1.351418
80,1.368880
90,1.301213
100,1.301548


TrainOutput(global_step=100, training_loss=1.451976375579834, metrics={'train_runtime': 165.6612, 'train_samples_per_second': 9.658, 'train_steps_per_second': 0.604, 'total_flos': 960833954664960.0, 'train_loss': 1.451976375579834, 'epoch': 1.3333333333333333})

## 18. Unsloth: быстрый inference

После обучения Unsloth рекомендует переключить модель в режим inference.

In [21]:
def generate_answer(model, tokenizer, instruction, max_new_tokens=160):
    """
    Генерирует ответ модели в chat-формате.

    Аргументы:
        model: causal language model.
        tokenizer: токенизатор модели.
        instruction: пользовательская инструкция.
        max_new_tokens: максимальное число новых токенов.

    Возвращает:
        Строку ответа.
    """
    messages = [
        {
            "role": "system",
            "content": "Ты полезный русскоязычный ассистент. Отвечай ясно, кратко и по делу.",
        },
        {
            "role": "user",
            "content": instruction,
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [22]:
FastLanguageModel.for_inference(unsloth_model)

print(generate_answer(
    unsloth_model,
    unsloth_tokenizer,
    "Сравни полное дообучение, LoRA и QLoRA.",
))

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Полное дообучение (full-branching) — это метод машинного обучения, который позволяет улучшить качество модели, используя потенциальные потенциалисты для дальнейшего обучения, пока модель не достигнет определенного уровня. Полное дообучение использует несколько порожденных моделей, чтобы улучшить их функциональность и обучаться вместе. Полное дообучение работает по следующему принципу: модель обучается на наборе примеров, которые она использует для обучения, но она не может обучаться на своих собственных данных, которые она не использует. Потенциалы модели могут быть отключены или ограничены, если они


# Часть C. Практические выводы

## Что выбрать

| Сценарий | Подход |
|---|---|
| Очень мало GPU памяти | QLoRA 4-bit |
| Нужно быстро обучить стиль ответа | LoRA / QLoRA |
| Нужно сильно изменить знания модели | больше данных, возможно full fine-tuning |
| Нужна максимальная скорость обучения | Unsloth |
| Нужен простой стандартный стек | Hugging Face + PEFT |

## Типичные гиперпараметры LoRA

| Параметр | Стартовое значение |
|---|---:|
| `r` | 8 или 16 |
| `lora_alpha` | 16 или 32 |
| `lora_dropout` | 0.05 |
| `learning_rate` | `1e-4` – `2e-4` |
| `max_length` | 512 – 2048 |
| `gradient_accumulation_steps` | 4 – 16 |

## 19. Частые ошибки

1. **Модель не помещается в GPU**  
   Уменьшите `MAX_LENGTH`, batch size, используйте 4-bit, включите gradient checkpointing.

2. **Модель учит prompt, а не ответ**  
   Для production используйте assistant-only loss masking.

3. **После обучения ответы стали хуже**  
   Проверьте качество датасета, learning rate, число эпох и формат chat template.

4. **Loss падает, но ответы плохие**  
   Loss не всегда отражает instruction-following качество. Нужны ручные тесты и eval-набор.

5. **Unsloth не устанавливается**  
   Проверьте версию Python, PyTorch, CUDA и рекомендации Unsloth для вашей платформы.

## 20. Итоги

В этом блокноте мы:

- разобрали fine-tuning, LoRA, QLoRA и квантизацию;
- загрузили русскоязычный instruction dataset;
- подготовили chat-format данные;
- загрузили маленькую Qwen-модель;
- применили 4-bit квантизацию;
- добавили LoRA через PEFT;
- запустили шаблон дообучения через Hugging Face Trainer;
- сохранили LoRA-адаптер;
- разобрали альтернативный пайплайн Unsloth.

Главная мысль:

> Для большинства учебных и прикладных экспериментов начинать стоит не с полного fine-tuning, а с QLoRA/LoRA на небольшой модели и чистом датасете.